# 02 — Feature Engineering
**Forecast horizon**: 24 hours ahead

CRITICAL CONSTRAINT: All features must use only information available at issue time **t**.
The target is pm25 at **t + 24h**.


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import yaml
from pathlib import Path

cfg = yaml.safe_load(open('../configs/config.yaml'))
df = pd.read_csv('../data/processed/hourly_processed.csv', index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index, utc=True)
df = df.sort_index()
print(f"Loaded: {df.shape}")


## Feature Matrix Construction

In [ ]:
from src.features.build_features import build_features, get_feature_cols, verify_target_alignment

lag_hours = cfg['features']['lag_hours']
rolling_windows = cfg['features']['rolling_windows']
horizon = cfg['target']['horizon_hours']

feat = build_features(df, lag_hours=lag_hours, rolling_windows=rolling_windows, horizon=horizon)
feature_cols = get_feature_cols(feat)

print(f"Feature matrix shape: {feat.shape}")
print(f"Number of predictor features: {len(feature_cols)}")
print()
print("Feature columns:")
for fc in feature_cols:
    print(f"  {fc}")


## Target Alignment Verification

In [ ]:
print("Verifying that target = pm25[t + 24h]...")
verify_target_alignment(df, horizon=horizon)
print()

# Manual spot-check
t = df.index[100]
t_future = t + pd.Timedelta(hours=24)
feat_row = feat.loc[t]
raw_val = df.loc[t_future, 'pm25']
feat_val = feat_row['target']
print(f"Issue time:   {t}")
print(f"Target time:  {t_future}")
print(f"feat['target'] = {feat_val:.4f}")
print(f"df['pm25'][t+24h] = {raw_val:.4f}")
print(f"Match: {abs(feat_val - raw_val) < 1e-6}")


## Sample Feature Matrix

In [ ]:
print("First 5 rows of feature matrix (non-null target rows only):")
display = feat.dropna(subset=['target']).head()
with pd.option_context('display.max_columns', 10, 'display.float_format', '{:.3f}'.format):
    print(display[['target', 'pm25_lag0', 'pm25_lag24h', 'pm25_roll24h_mean',
                    'target_hour', 'temp_c', 'rh', 'ws']].to_string())


## Missingness in Feature Matrix

In [ ]:
valid = feat.dropna(subset=['target'])
print(f"Rows with non-null target: {len(valid):,}")
print(f"Rows with all features non-null: {valid.dropna().shape[0]:,}")
print()
print("Missing % by feature:")
for col in feature_cols:
    pct = valid[col].isna().mean() * 100
    print(f"  {col:<30} {pct:.2f}%")
print()
print("Missing values handled by SimpleImputer(strategy='median') inside the sklearn Pipeline.")
print("The imputer is fitted ONLY on training data to prevent leakage.")


## Feature Engineering Summary

| Group | Features | Description |
|---|---|---|
| PM2.5 lags | pm25_lag0 through lag168h | Current and past PM2.5 at issue time t |
| Rolling mean | pm25_roll6h_mean, pm25_roll24h_mean | Short-term and daily average smoothing |
| Rolling std | pm25_roll24h_std | Variability measure |
| Calendar | target_hour, target_dow, target_month | Derived from target time t+24h |
| Cyclical | hour_sin, hour_cos | Smooth encoding of hour |
| Co-pollutants | pm10, no, no2, nox, nh3, so2, co, ozone | Measured at issue time t |
| Meteorology | temp_c, rh, ws, wd, rf, sr | Measured at issue time t |

**No future information** (from t+1 through t+24) is used in any feature.
